# Stage 4: Keyword Generation from Problems

In this stage, we'll learn how to:
1. Load problem descriptions from test dataset
2. Use LLM to extract 5-7 relevant keywords per problem
3. Combine domain concepts with technical Gurobi terms
4. Prepare queries for the retrieval stage

Key Concept: Instead of searching directly with the problem description,
we extract concise keywords that capture both:
- Domain context (e.g., "transportation", "profit maximization")
- Technical hints (e.g., "binary variables", "budget constraint")

This acts as "query expansion" before vector search.

Output: Keywords for each problem (displayed, not saved separately)

In [1]:
import pandas as pd

from config import (
    TEST_DATA_PATH,
    NUM_PROBLEMS,
    NUM_KEYWORDS_MIN,
    NUM_KEYWORDS_MAX,
    VERBOSE,
    OPENROUTER_MODEL
)
from llm_helper import get_llm

## SECTION 1: KEYWORD EXTRACTION PROMPT

In [2]:
KEYWORD_PROMPT_TEMPLATE = """You are a Gurobi and domain-focused optimization assistant.
You will receive a user's question about LP/ILP/MILP optimization problems.

Your task: Produce exactly {min_keywords}-{max_keywords} concise keywords or short phrases that capture both:
(1) The domain or scenario (such as "transportation," "investment," "resource allocation," "profit maximization")
(2) Relevant Gurobi or solver features (like "binary variables," "budget constraint," "addVar," "optimize," "addConstr," "setObjective," "Model")

Guidelines:
- Look for domain hints (e.g., fishery, orchard, printers, workforce planning)
- Look for optimization clues (e.g., "maximize", "minimize", "at most", "less than")
- Each keyword phrase: 1-3 words maximum (e.g., "transportation budget", "binary var constraints")
- Output exactly {min_keywords}-{max_keywords} items, comma-separated
- No extra commentary or headings
- Focus on terms that would appear in technical documentation

User's Question:
{QUESTION}

Keywords ({min_keywords}-{max_keywords}):"""

# Initialize LLM (OpenRouter with Gemini 2.5 Flash)
print(f"Initializing LLM: {OPENROUTER_MODEL}")
print("   (Connecting to OpenRouter API...)")
llm = get_llm()

Initializing LLM: google/gemini-2.5-flash
   (Connecting to OpenRouter API...)


## SECTION 2: KEYWORD GENERATION FUNCTION

In [3]:
def generate_keywords(llm, question: str) -> str:
    """
    Extract semantic keywords from a problem description.

    Args:
        llm: LLM instance
        question: Problem description

    Returns:
        Comma-separated keyword string

    Example Input:
        "A fishery wants to transport their catch. They can either use local
        sled dogs or trucks. Local sled dogs can take 100 fish per trip while
        trucks can take 300 fish per trip. The cost per trip for sled dogs is
        $50 while the cost per trip for a truck is $100. The budget is at most
        $1000 and the number of sled dog trips must be less than the number of
        truck trips. Formulate an LP to maximize the number of fish transported."

    Example Output:
        "fishery transportation, budget constraint, maximize fish, trip planning,
        cost minimization, resource allocation, addVar"

    Why This Helps:
        The keywords bridge the gap between colloquial problem descriptions
        and technical documentation. Terms like "budget constraint" and "addVar"
        will match relevant docs better than searching with the full paragraph.
    """
    prompt = KEYWORD_PROMPT_TEMPLATE.format(
        QUESTION=question,
        min_keywords=NUM_KEYWORDS_MIN,
        max_keywords=NUM_KEYWORDS_MAX
    )
    response = llm.invoke(prompt).strip()
    return response

## SECTION 3: EXECUTE PIPELINE

In [4]:
print("=" * 80)
print("STAGE 4: KEYWORD GENERATION FROM PROBLEMS")
print("=" * 80)

# Load test dataset
print(f"\nLoading test dataset: {TEST_DATA_PATH}")
df = pd.read_csv(TEST_DATA_PATH)
print(f"   Total problems in dataset: {len(df)}")
print(f"   Problems for tutorial: {NUM_PROBLEMS}")

# Select first N problems for demonstration
demo_problems = df.head(NUM_PROBLEMS)

print(f"\nGenerating keywords for {NUM_PROBLEMS} problems...")
print(f"   (Target: {NUM_KEYWORDS_MIN}-{NUM_KEYWORDS_MAX} keywords per problem)\n")

# Store results
results = []

# Process each problem
for i, row in demo_problems.iterrows():
    question = row['Question']
    expected_objective = row['Objective']

    # Generate keywords
    keywords = generate_keywords(llm, question)

    # Store result
    results.append({
        'problem_id': i,
        'question': question,
        'keywords': keywords,
        'expected_objective': expected_objective
    })

STAGE 4: KEYWORD GENERATION FROM PROBLEMS

Loading test dataset: /Users/tasnimahmed/Downloads/tutorial/Standalone/test_data.csv
   Total problems in dataset: 289
   Problems for tutorial: 2

Generating keywords for 2 problems...
   (Target: 5-7 keywords per problem)



In [5]:
print(f"{'=' * 80}")
print("DEMONSTRATION: Generated Keywords")
print(f"{'=' * 80}\n")

for result in results:
    print(f"{'─' * 80}")
    print(f"Problem #{result['problem_id']}")
    print(f"{'─' * 80}\n")

    # Show problem description (truncated)
    problem_text = result['question']
    if len(problem_text) > 300:
        problem_text = problem_text[:300] + "..."

    print("Problem Description:")
    print(f"   {problem_text}\n")

    print(f"Generated Keywords:")
    # Format keywords nicely
    keywords_list = [k.strip() for k in result['keywords'].split(',')]
    for j, kw in enumerate(keywords_list, 1):
        print(f"   {j}. {kw}")

    # print(f"\nExpected Objective Value: {result['expected_objective']}\n")

DEMONSTRATION: Generated Keywords

────────────────────────────────────────────────────────────────────────────────
Problem #0
────────────────────────────────────────────────────────────────────────────────

Problem Description:
   A fishery wants to transport their catch. They can either use local sled dogs or trucks. Local sled dogs can take 100 fish per trip while trucks can take 300 fish per trip. The cost per trip for sled dogs is $50 while the cost per trip for a truck is $100. The budget is at most $1000 and the number ...

Generated Keywords:
   1. Fishery Transportation
   2. Maximize Fish
   3. Budget Constraint
   4. Trip Constraints
   5. Continuous Variables
   6. Gurobi Model
   7. setObjective
────────────────────────────────────────────────────────────────────────────────
Problem #1
────────────────────────────────────────────────────────────────────────────────

Problem Description:
   An office supply company makes two types of printers: color printers and black and 

In [6]:
# Save results for next stage
print(f"\nSaving results for Stage 5...")
import pickle
with open('tutorial_keywords.pkl', 'wb') as f:
    pickle.dump(results, f)


Saving results for Stage 5...
